# Clase 119 — Losses, métricas, capas y modelos custom

Cuando los builtins de Keras no alcanzan, extendemos el framework: **loss**
custom (función o subclass), **métrica** stateful, **capa** custom con
`build`/`call`, y un **modelo** con `train_step` propio.

Requiere: `tensorflow` / `keras` (≥ 3.0).

## 1. Loss custom como función (Huber)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

def huber_loss(y_true, y_pred, delta=1.0):
    error = y_true - y_pred
    abs_error = tf.abs(error)
    cuadratica = 0.5 * tf.square(error)
    lineal = delta * (abs_error - 0.5 * delta)
    return tf.where(abs_error <= delta, cuadratica, lineal)

y_true = tf.constant([0.0, 1.0, 2.0, 5.0])
y_pred = tf.constant([0.2, 0.9, 4.0, 5.1])
print("Huber por sample:", huber_loss(y_true, y_pred).numpy())
print("Huber media:", float(tf.reduce_mean(huber_loss(y_true, y_pred))))

## 2. Loss custom como subclass (Focal Loss, Lin et al. 2017)

In [ ]:
class FocalLoss(keras.losses.Loss):
    """Focal Loss para clasificación con desbalance de clases."""

    def __init__(self, gamma=2.0, alpha=0.25, name="focal_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, y_pred.dtype)
        eps = keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
        ce = -y_true * tf.math.log(y_pred)
        peso = self.alpha * tf.pow(1.0 - y_pred, self.gamma)
        return tf.reduce_sum(peso * ce, axis=-1)

    def get_config(self):
        return {**super().get_config(), "gamma": self.gamma, "alpha": self.alpha}

fl = FocalLoss(gamma=2.0, alpha=0.25)
yt = tf.constant([[0.0, 1.0], [1.0, 0.0]])
yp = tf.constant([[0.1, 0.9], [0.8, 0.2]])
print("Focal loss por sample:", fl.call(yt, yp).numpy())

## 3. Métrica stateful custom (`update_state` / `result` / `reset_state`)

In [ ]:
class F1Macro(keras.metrics.Metric):
    """F1 (clasificación binaria) acumulando TP/FP/FN a lo largo del epoch."""

    def __init__(self, name="f1", **kwargs):
        super().__init__(name=name, **kwargs)
        self.tp = self.add_weight(name="tp", initializer="zeros")
        self.fp = self.add_weight(name="fp", initializer="zeros")
        self.fn = self.add_weight(name="fn", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)
        self.tp.assign_add(tf.reduce_sum(y_true * y_pred))
        self.fp.assign_add(tf.reduce_sum((1 - y_true) * y_pred))
        self.fn.assign_add(tf.reduce_sum(y_true * (1 - y_pred)))

    def result(self):
        eps = keras.backend.epsilon()
        precision = self.tp / (self.tp + self.fp + eps)
        recall = self.tp / (self.tp + self.fn + eps)
        return 2 * precision * recall / (precision + recall + eps)

    def reset_state(self):
        for v in self.variables:
            v.assign(0.0)

m = F1Macro()
m.update_state([1, 0, 1, 1], [0.9, 0.2, 0.4, 0.8])
print("F1 tras batch 1:", float(m.result()))
m.reset_state()
print("F1 tras reset_state:", float(m.result()))

## 4. Capa custom (`build` declara pesos, `call` hace el forward)

In [ ]:
class L2Normalize(keras.layers.Layer):
    """Normaliza cada vector de la última dimensión a norma unitaria."""

    def __init__(self, axis=-1, **kwargs):
        super().__init__(**kwargs)
        self.axis = axis

    def call(self, inputs):
        return tf.math.l2_normalize(inputs, axis=self.axis)


class DenseCustom(keras.layers.Layer):
    """Dense manual: los pesos se crean en build (shape dinámica)."""

    def __init__(self, units, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer="glorot_uniform", trainable=True, name="kernel")
        self.b = self.add_weight(
            shape=(self.units,), initializer="zeros", trainable=True, name="bias")

    def call(self, inputs):
        return self.activation(tf.matmul(inputs, self.w) + self.b)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"units": self.units,
                    "activation": keras.activations.serialize(self.activation)})
        return cfg

x = tf.random.normal((4, 8))
capa = DenseCustom(3, activation="relu")
print("salida DenseCustom:", capa(x).shape)
print("normas tras L2Normalize:", tf.norm(L2Normalize()(x), axis=-1).numpy())

## 5. `get_config` para serializar con `model.save()`

In [ ]:
modelo = keras.Sequential([
    keras.Input((8,)),
    DenseCustom(16, activation="relu"),
    DenseCustom(1, activation="sigmoid"),
])
modelo.save("modelo_custom.keras")
recargado = keras.models.load_model(
    "modelo_custom.keras", custom_objects={"DenseCustom": DenseCustom})
print("recargado OK | params:", recargado.count_params())

## 6. Modelo custom con `train_step` propio (gradient clipping manual)

In [ ]:
class ModeloConClip(keras.Model):
    """train_step custom: aplica clip_by_norm a los gradientes en cada batch."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.d1 = layers.Dense(64, activation="relu")
        self.out = layers.Dense(10, activation="softmax")

    def call(self, x):
        return self.out(self.d1(x))

    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            y_pred = self(x, training=True)
            loss = self.compute_loss(y=y, y_pred=y_pred)
        grads = tape.gradient(loss, self.trainable_variables)
        grads = [tf.clip_by_norm(g, 1.0) for g in grads]   # clipping manual
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        for metrica in self.metrics:
            if metrica.name == "loss":
                metrica.update_state(loss)
            else:
                metrica.update_state(y, y_pred)
        return {m.name: m.result() for m in self.metrics}

modelo = ModeloConClip()
modelo.compile(optimizer="adam",
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("modelo con train_step custom listo para fit()")

## Ejercicios

1. **Loss custom**: implementá **Focal Loss** y comparala contra cross-entropy
   en un dataset desbalanceado (clase minoritaria al 5 %); reportá F1 macro.
2. **Métrica F1**: heredá `keras.metrics.Metric` acumulando TP/FP/FN y usala
   durante `fit`; verificá que `reset_state` limpia entre épocas.
3. **Capa custom**: escribí `L2Normalize` y comprobá que la norma de cada
   vector de salida es 1.
4. **`train_step` custom**: subclaseá `keras.Model` aplicando gradient clipping
   manual y logging extra por batch.
5. **`get_config`**: agregalo a `DenseCustom` y verificá que `model.save()` +
   `load_model(custom_objects=...)` funciona.

## Conclusiones

- Una **loss** puede ser una función `(y_true, y_pred) -> tensor` o una subclass de `keras.losses.Loss` (con estado/config).
- Las métricas **stateful** (`update_state`/`result`/`reset_state`) son obligatorias para F1, precision, recall o AUC, que acumulan por época.
- En una **capa** custom, `build` declara los pesos con `add_weight` (shape dinámica) y `call` define el forward.
- Overridando `train_step` se personaliza el batch (clipping, dos optimizadores) sin perder `fit`, callbacks ni métricas.
- `get_config` es lo que habilita `model.save()` / `load_model(custom_objects=...)`.